# Hallucination Detection Task Example

The primary objective of this task is to develop a classification system capable of verifying the factual faithfulness of AI-generated clinical summaries against original medical records. Using the MIMIC-IV dataset, specifically focusing on "Brief Hospital Course" notes, we pair clinical source text with generated summaries to detect if the generated summaries have hallucinations (incorrect statements) or not. This gives a critical requirement for deploying Large Language Models (LLMs) in medical environments, ensuring that AI documentation doesn't contain incorrect or guessed details. 

This implementation is directly implemented by the ideas from the research paper by Hegselmann et al. (2023), which enforces the additional requirements of Large Language Models for clinical text summarization. By following their benchmarks for clinical accuracy and identifying common failure modes in medical LLMs, we have structured our task to address the most clinically significant errors. This example serves as a practical application of their findings, focusing on the detection of hallucinations they identified as barriers to real world medical deployment.

To confirm that the model is performing genuine factual verification rather than simply judging the writing style or fluency of the summaries, we conducted a **Source-Summary Mismatch Ablation**. In this experiment, we intentionally paired clinical notes from one patient with summaries belonging to an entirely different, unrelated patient. This "blind" test allows us to measure the model's sensitivity to context; if the model is truly grounded in the source text, it should easily identify these mismatched pairs as hallucinations. This approach helps ensure that our classifier is actively cross-referencing the clinical evidence rather than being overtrained on other specific information to make its predictions.

In [1]:
import re
import json
import string
import random

from copy import deepcopy
from datetime import datetime
from random import choice, shuffle
from typing import List, Optional, Any, Callable, Tuple, Dict

import torch
import numpy as np
import polars as pl

from pyhealth.data import Patient
from pyhealth.trainer import Trainer
from pyhealth.models import Transformer
from sklearn.metrics import accuracy_score
from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.tasks.hallucination_detection import HallucinationDetectionTask

C:\Users\kesav\Desktop\Masters\598D\Project\GIT\PyHealth\pyhealth\trainer.py:12: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import trange


## Set Seeds

Setting the seeds so we can see the same results in every iteration.

In [2]:
def set_all_seeds(seed: int) -> None:
    """Sets the seed for all relevant libraries to ensure reproducibility.

    Example:
        >>> set_all_seeds(42)

    Args:
        seed: The integer value to use as the random seed.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_all_seeds(12)

## Data Path

Change the pathway of the data files to your local path

In [3]:
"""Data path configurations for hallucination detection."""

# Path to Medical Expert Annotations of Unsupported Facts
# Reference: https://physionet.org/content/ann-pt-summ/1.0.0/

HALLUCINATION_DATA_PATH: str = (
    "../../Data/ann-pt-summ-1.0.0/hallucination_datasets/hallucinations_mimic_di.jsonl"
)
NON_HALLUCINATION_DATA_PATH: str = (
    "../../Data/ann-pt-summ-1.0.0/mimic-iv-note-ext-di-bhc/dataset/train_100.json"
)

## Clean Data

To prepare the clinical text for the model, we first normalize the raw notes by lowercasing, removing special characters, and stripping punctuation to create a clean tokenized word list. Given that clinical notes can be extremely lengthy, we implement an anchored cropping strategy to fit the text within the model's context window. For instances flagged as hallucinations, the logic identifies a "grounding anchor" word from the error labels and centers a 100-word window around it. This ensures that the specific clinical evidence needed to identify a hallucination is preserved, providing a high signal-to-noise ratio for the classifier.

In [4]:
def get_clean_word_list(text: str) -> List[str]:
    """Cleans text by removing punctuation and normalizing whitespace.

    Args:
        text: The raw string to be cleaned.

    Returns:
        A list of lowercase words stripped of punctuation.
    """
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[\[\]_]', '', text)
    punctuation_set = set(string.punctuation)
    clean_text = "".join(char for char in text if char not in punctuation_set)
    return clean_text.split()

def crop_to_window(
    word_list: List[str], anchor_word: Optional[str] = None, window_size: int = 100
) -> List[str]:
    """Crops a list of words to a fixed window size, centering on an anchor.

    If the anchor_word is found, the window is centered around its first 
    occurrence. If not found or not provided, the window starts from the 
    beginning of the list.

    Example:
        >>> words = ["the", "patient", "has", "a", "severe", "headache"]
        >>> crop_to_window(words, anchor_word="has", window_size=4)
        ['patient', 'has', 'a', 'severe']

    Args:
        word_list: The full list of words to crop.
        anchor_word: The word used to center the window. Defaults to None.
        window_size: The maximum number of words to return. Defaults to 100.

    Returns:
        A sub-list of words containing up to window_size elements.
    """
    if not word_list:
        return []

    start_idx = 0
    # If there is a hallucination word, find its position
    if anchor_word and anchor_word.lower() in word_list:
        anchor_idx = word_list.index(anchor_word.lower())
        # Center the anchor
        start_idx = max(0, anchor_idx - (window_size // 2))

    # Ensure we don't go out of bounds at the end
    end_idx = min(len(word_list), start_idx + window_size)
    
    # Adjust start if we hit the end to maintain window_size if possible
    if end_idx == len(word_list):
        start_idx = max(0, end_idx - window_size)

    return word_list[start_idx:end_idx]

def load_jsonl_to_samples(
    file_path: str, task_executor: Callable, max_samples: int = 100
) -> List[Any]:
    """Loads JSONL data and converts it into PyHealth-compatible samples.

    Args:
        file_path: Path to the .jsonl file.
        task_executor: The PyHealth task function/class to process patients.
        max_samples: Maximum number of samples to collect. Defaults to 100.

    Returns:
        A list of processed samples ready for a DataLoader.
    """
    samples = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            data = json.loads(line)
            
            label = 1 if len(data.get("labels", [])) > 0 else 0
            
            # Get full clean word lists
            full_source_words = get_clean_word_list(data["text"])
            full_summary_words = get_clean_word_list(data["summary"])
            
            # Find an anchor word if it's a hallucination
            anchor = None
            if label == 1:
                # Assuming 'labels' contains the hallucinated text/words
                # We take the first one as our 'grounding' anchor
                raw_label_info = data["labels"][0]
                # If your label is a dict with 'text', use that; if it's a string, use
                #  it directly
                if isinstance(raw_label_info, dict):
                    anchor_text = raw_label_info['text'] 
                else: 
                    anchor_text = str(raw_label_info)
                # Take the first word of the hallucination
                anchor = anchor_text.split()[0] 

            # CROP BOTH TO 100
            cropped_source = crop_to_window(
                full_source_words, 
                anchor_word=anchor, 
                window_size=100
                )
            
            # Summaries are usually short anyway
            cropped_summary = full_summary_words[:100] 

            p_id = f"P_{i}"
            v_id = f"V_{i}"
            
            patient_df = pl.DataFrame({
                "timestamp": [datetime(2026, 1, 1), datetime(2026, 1, 2)],
                "event_type": ["noteevents", "visits"],
                "noteevents/text": [cropped_source[:50], None],
                "noteevents/visit_id": [v_id, None],
                "visits/visit_id": [None, v_id],
                "visits/ai_summary": [None, cropped_summary[:50]],
                "visits/hallucination_label": [None, label]
            })

            patient = Patient(patient_id=p_id, data_source=patient_df)
            samples.extend(task_executor(patient))

            if len(samples) >= max_samples:
                break
                
    return samples

In [5]:
def get_data(train_limit: int = 150) -> Tuple[List[Any], List[Any], List[Any]]:
    """Loads, shuffles, and splits the hallucination detection dataset.

    Example:
        >>> all_s, train_s, test_s = get_data()

    Returns:
        A tuple containing (all_samples, train_samples, test_samples).
    """

    print("Loading Training Data...")
    task = HallucinationDetectionTask()

    samples = load_jsonl_to_samples(
        HALLUCINATION_DATA_PATH, 
        task
    )

    samples_non_hallucination = load_jsonl_to_samples(
        NON_HALLUCINATION_DATA_PATH, 
        task
    )

    all_samples = samples + samples_non_hallucination
    shuffle(all_samples)

    train_samples, test_samples = all_samples[:train_limit], all_samples[train_limit:]

    print(f"\nSetup Complete:")
    print(f"Train size: {len(train_samples)}")
    print(f"Test size:  {len(test_samples)}")

    return all_samples, train_samples, test_samples

all_samples, train_samples, test_samples = get_data()

Loading Training Data...

Setup Complete:
Train size: 150
Test size:  50


## Model Architecture and Training Pipeline

To perform the classification, we utilize the PyHealth Transformer model, which is specifically designed to handle multi-stream clinical sequences. The model architecture is configured with a 64-dimensional embedding space and a multi-head attention mechanism (2 heads, 2 layers). This setup allows the model to learn complex dependencies between the consolidated clinical notes in the source_text and the AI-generated summary_text.

The training process is managed via the Trainer class, using the ROC-AUC metric as the primary monitor for convergence. By passing our custom input_schema (defining both text sources as "sequence") and output_schema (defining the label as "binary") into the create_sample_dataset helper, we ensure that the Transformer's embedding layers are automatically mapped to the vocabulary generated from our cleaned tokens. This seamless integration between the data processing task and the modeling stage allows for efficient end-to-end training of the hallucination detector.

In [6]:
def create_train_model(
    samples: List[Dict[str, Any]], 
    input_schema: Dict[str, str], 
    output_schema: Dict[str, str], 
    device: str = "cpu", 
    epochs: int = 5
) -> Trainer:
    """Initializes and trains a Transformer model on the provided samples.

    Example:
        >>> trainer = create_train_model(samples, in_schema, out_schema)

    Args:
        samples: List of PyHealth-formatted sample dictionaries.
        input_schema: Dictionary mapping feature names to PyHealth types.
        output_schema: Dictionary mapping label names to PyHealth types.
        device: Device to run training on ("cpu" or "cuda"). Defaults to "cpu".
        epochs: Number of training iterations. Defaults to 5.

    Returns:
        The trained PyHealth Trainer instance.
    """

    # 1. Create the SampleDataset
    dataset = create_sample_dataset(
        samples=samples,
        input_schema=input_schema,
        output_schema=output_schema,
        dataset_name="clinical_summary_eval",
    )

    # 2. Initialize the Transformer
    # The model uses the dataset's internal metadata to configure its feature streams
    model = Transformer(
        dataset=dataset,
        embedding_dim=64,
        heads=2,
        num_layers=2,
        max_seq_len=200
    )

    # 3. Run a forward pass
    loader = get_dataloader(dataset, batch_size=32, shuffle=True)
    batch = next(iter(loader))
    output = model(**batch)

    # 4. Train the model on the training dataset
    trainer = Trainer(model=model, device=device, enable_logging=False)
    trainer.train(
        train_dataloader=loader,
        epochs=epochs,
        monitor="roc_auc"
    )

    return trainer


# Global definitions with type hints
input_schema: Dict[str, str] = {
    "source_text": "sequence", 
    "summary_text": "sequence"
}
output_schema: Dict[str, str] = {
    "label": "binary"
}
trainer = create_train_model(train_samples, input_schema, output_schema)

Label label vocab: {0: 0, 1: 1}
Transformer(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (source_text): Embedding(1781, 64, padding_idx=0)
    (summary_text): Embedding(1251, 64, padding_idx=0)
  ))
  (transformer): ModuleDict(
    (source_text): TransformerLayer(
      (transformer): ModuleList(
        (0-1): 2 x TransformerBlock(
          (attention): MultiHeadedAttention(heads=2, d_model=64, dropout=0.1)
          (feed_forward): PositionwiseFeedForward(
            (w_1): Linear(in_features=64, out_features=256, bias=True)
            (w_2): Linear(in_features=256, out_features=64, bias=True)
            (dropout): Dropout(p=0.5, inplace=False)
            (activation): GELU(approximate='none')
          )
          (input_sublayer): SublayerConnection(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (dropout): Dropout(p=0.5, inplace=False)
          )
          (output_sublayer): SublayerConnection(
            (norm)

Epoch 0 / 5: 100%|██████████| 5/5 [00:00<00:00,  8.09it/s]

--- Train epoch-0, step-5 ---
loss: 0.9192




Epoch 1 / 5: 100%|██████████| 5/5 [00:00<00:00,  8.92it/s]

--- Train epoch-1, step-10 ---
loss: 0.7744



...

--- Train epoch-2, step-15 ---
loss: 0.7949




Epoch 3 / 5: 100%|██████████| 5/5 [00:00<00:00,  9.02it/s]

--- Train epoch-3, step-20 ---
loss: 0.8470




Epoch 4 / 5: 100%|██████████| 5/5 [00:00<00:00,  8.88it/s]

--- Train epoch-4, step-25 ---
loss: 0.7921


## Model Evaluation and Baseline Performance

After training the Transformer, we evaluated its performance on a held-out test set using the trainer.inference pipeline. The model achieved a 74% test accuracy in identifying hallucinations. This baseline performance indicates that the model has successfully learned to recognize clinical inconsistencies between the source notes and the summaries, despite the nuanced nature of medical language. This result is particularly significant given the complexity of the task, as it demonstrates that the classifier is moving beyond random guessing and is successfully capturing semantic patterns that distinguish faithful clinical summaries from those containing fabricated or incorrect information.

In [7]:
def test_accuracy(
    trainer: Trainer, 
    test_samples: List[Dict[str, Any]], 
    input_schema: Dict[str, str], 
    output_schema: Dict[str, str],
) -> None:
    """Evaluates the model accuracy on a test dataset.

    Example:
        >>> test_accuracy(test_samples, in_schema, out_schema, trainer)

    Args:
        trainer: The PyHealth Trainer instance used for inference.
        test_samples: List of samples to evaluate.
        input_schema: Dictionary defining the input features.
        output_schema: Dictionary defining the output labels.
    """

    test_dataset = create_sample_dataset(
        samples=test_samples,
        input_schema=input_schema,
        output_schema=output_schema
    )
    test_loader = get_dataloader(test_dataset, batch_size=20, shuffle=False)

    y_true, y_prob, _ = trainer.inference(test_loader)

    y_pred = (y_prob > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)

    print(f"Calculated Accuracy: {acc:.4f}")

test_accuracy(trainer, test_samples, input_schema, output_schema)

Label label vocab: {0: 0, 1: 1}


Evaluation: 100%|██████████| 3/3 [00:00<00:00, 81.78it/s]

Calculated Accuracy: 0.7400


## Source-Summary Mismatch Ablation

To verify the model's robustness, we conducted an ablation study using a Source-Summary Mismatch test. By randomly swapping the source_text between different patients while labeling them as hallucinations, we created a dataset where the summary is entirely unsupported by the provided clinical note. The model achieved a significantly higher accuracy of 92% on this ablated data. This jump from our baseline accuracy (74%) confirms that the model is not merely memorizing summary styles; instead, it has learned to actively cross-reference the content of the summary against the source text. The high performance on mismatched pairs demonstrates that the Transformer is highly sensitive to the lack of mutual information, effectively proving that the model is "grounded" in the provided evidence.

In [8]:
def create_ablation_dataset(
    all_samples: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """Creates an ablation dataset by randomly swapping source texts.

    This helps verify if the model is truly sensitive to the relationship 
    between the source and the summary.

    Args:
        all_samples: The original list of sample dictionaries.

    Returns:
        A list of 100 samples with mismatched source/summary pairs.
    """

    all_samples_hallucinated = deepcopy(all_samples)

    for d in all_samples_hallucinated:
        d['source_text'] = choice(all_samples)['source_text']
        d['label'] = 1
    all_samples_hallucinated[0]['label'] = 0

    return all_samples_hallucinated[:100]

all_samples_hallucinated = create_ablation_dataset(all_samples)
test_accuracy(trainer, all_samples_hallucinated, input_schema, output_schema)

Label label vocab: {0: 0, 1: 1}


Evaluation: 100%|██████████| 5/5 [00:00<00:00, 94.82it/s]

Calculated Accuracy: 0.9200
